# Forecasting Daily Sales with LightGBM — A Walkthrough

This notebook is a worked example of a **tabular time-series forecasting** workflow built around the
[Rohlik Orders Forecasting Challenge](https://www.kaggle.com/competitions/rohlik-sales-forecasting-challenge-v2)
(an online grocery delivery dataset). We forecast the number of units sold for each `(product, warehouse, day)`
combination, given history of sales, prices, discounts, and a calendar of holidays.

The notebook is intentionally structured the way many top Kaggle solutions are:

1. **Load and join the raw tables** — sales, inventory metadata, calendar.
2. **Engineer features** — calendar, price, hierarchy, rolling/EMA windows.
3. **Build target-derived features carefully** — these can leak the answer if you're sloppy.
4. **Train a LightGBM regressor** with k-fold cross-validation, early stopping, and a power transform on the target.
5. **Average out-of-fold predictions** for an honest validation score and **average test predictions** across folds for a more stable submission.

## Why LightGBM for time-series?

When practitioners hear "time series" they often jump to ARIMA, Prophet, or RNNs/transformers. But on **panel data** —
where you have many series sharing structure (here: thousands of products × warehouses) and rich exogenous
features (price, discount, calendar) — **gradient-boosted decision trees** like LightGBM, XGBoost, and CatBoost
are extremely strong baselines and frequently win these competitions. They:

- handle mixed numeric/categorical features natively (no one-hot blow-up),
- are robust to missing values and feature scale,
- don't require stationarity,
- and can pool information across all series in one model.

The trade-off is that **a tree model does not learn temporal patterns on its own**. Lags, rolling means,
seasonality, and cycles must be **engineered as features**. That's why ~80% of this notebook is feature
engineering and only ~20% is model training.

## Conventions

- `X_*` and `y_*` follow scikit-learn naming.
- Functions starting with `fe_` build features in-place on a DataFrame.
- A "fold" refers to one split of `RepeatedKFold` cross-validation.
- "OOF" = **out-of-fold** prediction: the model's prediction for a row when that row was *not* in its training fold.


## 1. Imports

A small dependency footprint — `numpy`/`pandas` for data wrangling, `scikit-learn` for the metric and
cross-validation splitter, and `lightgbm` for the model itself. `deepcopy` is used later to keep the
original train/test frames untouched while we mutate copies inside the CV loop.


In [ ]:
import numpy as np
import pandas as pd
from copy import deepcopy
from sklearn.metrics import mean_absolute_error      # Competition metric is weighted MAE.
from sklearn.model_selection import RepeatedKFold    # K-fold CV; see caveat about time-series below.
from lightgbm import LGBMRegressor, early_stopping, log_evaluation


## 2. Feature engineering functions

We split the feature engineering into three functions, applied at different points in the pipeline:

| Function | When applied | Why separate |
|---|---|---|
| `fe_date(df)` | After loading train/test | Pure functions of `date`; can run on either frame independently. |
| `fe_other(df)` | After loading train/test | Mostly per-row transforms + same-day group aggregations. |
| `fe_combined(df)` | After concatenating train+test | Needs cross-frame statistics (e.g. per-product mean price across all time). |

### Why `fe_combined` runs on `train + test`

For features like the per-product mean price, computing them on `train` alone would leave NaNs for any
product that only appears in `test`, and would also give the model a slightly different distribution at
inference time. Concatenating train+test for **non-target-derived** features is a standard, **safe** trick
in Kaggle pipelines: we are only using `X` columns from `test`, never `y`.

### Cyclical encoding of the day-of-year

`day_of_year` ranges 1..365. If we feed it raw, the model treats Dec 31 (day 365) and Jan 1 (day 1) as
maximally far apart, when in reality they are adjacent. The fix is to project the integer onto a circle:

$$
\text{cos\_day} = \cos\!\left(\frac{2\pi \cdot \text{day\_of\_year}}{365}\right), \quad
\text{sin\_day} = \sin\!\left(\frac{2\pi \cdot \text{day\_of\_year}}{365}\right)
$$

Together `(sin_day, cos_day)` uniquely identify a day-of-year while making the topology continuous —
the model can now learn "things near year-end behave like things near year-start." Useful for any
seasonal feature (hour-of-day, day-of-week, month, etc.).

> **Note for trees specifically:** Trees don't *need* monotonic transforms (they only care about ordering),
> but cyclical encoding gives them two split-able coordinates capturing seasonality cleanly. The `day_of_week`
> column is left as integer 0–6 — for a small range, trees can split it directly.

### Hierarchical aggregations (`common_name`)

Product names look like `"butter_premium_500g"`. Extract everything before the first `_`
as a `common_name` (think: product family). Features built by grouping on `(date, warehouse, common_name)`
let one product borrow strength from siblings in the same family on the same day. This is a form of
**target/group encoding** when the target is the aggregator, but here we're aggregating *features*
(discount levels, count of variants), which is leakage-free.

### Log-transforming `sell_price_main`

LightGBM is invariant to monotonic transforms of any single feature. So `log(price)` *by itself* changes nothing. But once you build
**derived features** on top of price (differences, ratios, scaled versions in `fe_combined`),
those derived features *are* affected. Log-scaling makes price differences additive in log-space,
which is how prices actually move (e.g. a 10% increase looks the same regardless of base price).

### `fe_combined` — features that need both train and test

- **`num_sales_days_28D`**: rolling count of how many days the product appeared in the last 28 days.
  Captures product "freshness" / how recently it's been on shelves. `closed='left'` is critical:
  it excludes the current row from the window so we never use today to predict today.
- **`price_detrended`**: a within-product z-score of price (`price_scaled`), then we subtract the
  daily-warehouse mean of that z-score. Prices drift up
  over time, and "is this item cheap *for this product, accounting for general inflation*" is a
  sharper signal than raw price.
- **`mean_orders_14d` / `ewmean_orders_56`**: rolling and exponentially-weighted means of total
  warehouse-level orders. These act as **broad demand indicators** — when total orders surge, individual
  products' sales tend to surge too. Using the *median* across products inside each warehouse before
  smoothing makes it robust to product-level outliers (a single hot SKU won't dominate).


In [ ]:
def fe_date(df):
    # Pure date features: year, day-of-week, days since an arbitrary anchor (linear time trend),
    # and (sin, cos) of day-of-year for cyclical seasonality.
    df['year'] = df['date'].dt.year
    df['day_of_week'] = df['date'].dt.dayofweek
    df['days_since_2020'] = (df['date'] - pd.to_datetime('2020-01-01')).dt.days.astype('int')  # monotonic time trend
    df['day_of_year'] = df['date'].dt.dayofyear
    df['cos_day'] = np.cos(df['day_of_year']*2*np.pi/365)  # Dec 31 ~ Jan 1 in (cos, sin) space.
    df['sin_day'] = np.sin(df['day_of_year']*2*np.pi/365)

def fe_other(df):
    # The dataset records up to 7 different "discount types" per row.
    discount_cols = ['type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount']
    df[discount_cols] = df[discount_cols].clip(0)  # Replace any negative entries with 0 (data hygiene).
    # max_discount = the strongest promotion across types 0–5 on this row. (type_6 is excluded here on purpose.)
    df['max_discount'] = df[['type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount']].max(axis=1)

    # Given that we're using LightGBM, which is in theory invariant to monotonic transformations of features, this transformation in isolation doesn't really do anything. I mainly did it because it made the shap plot look more linear. However, I think it did make further feature engineering that used price more effective.
    df['sell_price_main'] = np.log(df['sell_price_main'])

    # Names look like "<family>_<variant>_<size>". Take everything before the first "_" as a coarse family key.
    df['common_name'] = df['name'].apply(lambda x: x[:x.find('_')])
    # Same-day group aggregations. transform() broadcasts the group result back to each row.
    df['CN_total_products'] = df.groupby(['date','warehouse','common_name'])['unique_id'].transform('nunique')   # how many siblings are on shelves today
    df['CN_discount_avg']   = df.groupby(['date','warehouse','common_name'])['max_discount'].transform('mean')   # avg promo intensity in the family today
    df['CN_WH'] = df['common_name'] + '_' + df['warehouse']                                                       # categorical interaction: family × warehouse
    df['name_num_warehouses'] = df.groupby(['date','name'])['unique_id'].transform('nunique')                    # geographic breadth of this exact SKU today

def fe_combined(df):
    # Rolling 28-day count of days this product appeared. closed='left' = window is [t-28, t-1], excludes today.
    # This is "how long has this product been around / how regularly does it appear".
    df['num_sales_days_28D'] = pd.MultiIndex.from_frame(df[['unique_id','date']]).map(df.sort_values('date').groupby('unique_id').rolling(
        window='28D', on='date', closed='left')['date'].count().fillna(0))

    # This 'price_detrended' feature was one I found pretty late into the game, but I think it helped out a lot. I was trying to make a feature that captured whether an item was cheap or expensive relative to its usual price, which is what 'price_scaled' represents. What I found was that the prices of things generally increase over time. So I removed that time-based trend to construct price_detrended, and that proved very effective.
    mean_prices = df.groupby(df['unique_id'])['sell_price_main'].mean()
    std_prices  = df.groupby(df['unique_id'])['sell_price_main'].std()
    # Per-product z-score of (log) price. np.where guards against std==0 (constant-price products) → produce 0.
    df['price_scaled'] = np.where(df['unique_id'].map(std_prices) == 0, 0,
                                  (df['sell_price_main'] - df['unique_id'].map(mean_prices))/df['unique_id'].map(std_prices))
    # Subtract the daily-warehouse mean of price_scaled to remove the global "prices drift up over time" trend.
    df['price_detrended'] = df['price_scaled'] - df.groupby(['days_since_2020','warehouse'])['price_scaled'].transform('mean')
    df.drop('price_scaled',axis=1,inplace=True)  # only keep the detrended version

    # Warehouse-level demand index. Use median across products inside each (date, warehouse) for robustness,
    # then smooth with a 14-day rolling mean and a 56-day EWM. These are exogenous proxies for
    # "is the whole warehouse busy today?".
    warehouse_stats = df.groupby(['date','warehouse'])['total_orders'].median().rename('med_total_orders').reset_index().sort_values('date')
    warehouse_stats['ewmean_orders_56'] = warehouse_stats.groupby('warehouse')['med_total_orders'].transform(lambda x:x.ewm(alpha=1/56).mean())
    df['mean_orders_14d'] = pd.MultiIndex.from_frame(df[['warehouse','date']]).map(
        warehouse_stats.groupby('warehouse').rolling(on='date',window='14D')['med_total_orders'].mean())
    df['ewmean_orders_56'] = pd.MultiIndex.from_frame(df[['warehouse','date']]).map(
        warehouse_stats.set_index(['warehouse','date'])['ewmean_orders_56'])
    return df


## 3. Load the inventory (product metadata) table

`inventory.csv` is a *static* table: one row per product, listing its name and category hierarchy
(L1 → L4). It does **not** vary over time, so we'll merge it into the sales tables later as a
left-join on `unique_id`.

We drop two columns up front:

- `warehouse` — already present in the sales tables.
- `product_unique_id` — a redundant identifier; `unique_id` is the one we'll key on.

Dropping early prevents accidentally creating duplicate `_x`/`_y` suffixes during merges.


In [ ]:
inventory = pd.read_csv('inventory.csv').drop(['warehouse','product_unique_id'],axis=1)
inventory.head()


## 4. Calendar features — proximity to holidays

Retail demand is **violently affected by holidays** — both the day itself and the days just before/after.
A standard trick is to encode, for each row:

- whether today is a holiday,
- how many days since the last holiday,
- how many days until the next one.

This notebook uses a clean **forward-fill / back-fill** pattern to compute those distances:

1. Set `last_holiday_date = next_holiday_date = date` only on holiday rows; NaN elsewhere.
2. `ffill` within each warehouse → on any row, `last_holiday_date` carries forward the most recent holiday's date.
3. `bfill` similarly → `next_holiday_date` carries backward the upcoming holiday's date.
4. Subtract from `date` to get the integer distances.

In the final code, we keep only the binary "day before / day after" flags and drop the raw
distances.

The line `calendar.loc[calendar['holiday_name'].isna(), 'holiday'] = 0` corrects rows where the
`holiday` flag was set but `holiday_name` is missing (likely an upstream data inconsistency).


In [ ]:
calendar = pd.read_csv('calendar.csv', parse_dates=['date'])
calendar.loc[calendar['holiday_name'].isna(), 'holiday'] = 0  # V3: fix mislabeled holidays with no name

# Mark today's date in two helper columns ONLY on actual holiday rows; everything else NaN.
calendar['last_holiday_date'] = calendar['date']
calendar['next_holiday_date'] = calendar['date']
calendar.loc[calendar['holiday'] == 0, ['last_holiday_date','next_holiday_date']] = np.nan

# Within each warehouse, propagate the last/next holiday dates forward/backward in time.
calendar['last_holiday_date'] = calendar.sort_values('date').groupby('warehouse')['last_holiday_date'].ffill()
calendar['next_holiday_date'] = calendar.sort_values('date').groupby('warehouse')['next_holiday_date'].bfill()

# Compute integer day distances and derive the two binary flags we'll actually keep.
calendar['days_since_last_holiday'] = ((calendar['date'] - calendar['last_holiday_date']).dt.days)
calendar['days_to_next_holiday']    = ((calendar['next_holiday_date'] - calendar['date']).dt.days)
calendar['day_before_holiday'] = calendar['days_to_next_holiday'] == 1
calendar['day_after_holiday']  = calendar['days_since_last_holiday'] == 1

# Drop scaffolding + columns the author found uninformative in this competition.
calendar.drop(['last_holiday_date','next_holiday_date'],axis=1,inplace=True)
calendar.drop(['days_since_last_holiday','days_to_next_holiday'],axis=1,inplace=True)
calendar.drop(['shops_closed','winter_school_holidays','school_holidays','holiday_name'],axis=1,inplace=True)


## 5. Assemble the train/test datasets

We now bring the four sources together:

```
sales_train.csv ─┐
sales_test.csv  ─┼── join on (unique_id) → inventory metadata
                 └── join on (date, warehouse) → calendar features
```

A few things worth noticing:

- **The compound `id` index**: `f"{unique_id}_{date}"` gives every row a unique label. It's used to
  re-align frames after merges (which scramble row order) via `.loc[train.index]`.
- **`merge(...).set_index('id').loc[train.index]`** — a clean idiom for "join, then restore the
  original row order." Order doesn't usually matter for ML, but it does for diagnostics, exports,
  and ensures train and test stay deterministic.
- **`fe_combined` runs on `pd.concat([train, test])`** because some of its features (per-product
  mean price, the warehouse demand EWMs) need both halves to be computed correctly. We then
  split back into `train` and `test` with `.loc[]`. This is safe because `fe_combined`
  **does not touch `sales`** — there's no target leakage.
- **`test` drops `sales` and `availability`**: `sales` is what we predict; `availability` only
  exists in the historical record (we don't know whether something was in stock on a future
  day, so the model must not rely on it).


In [ ]:
# ---- TRAIN ----
train = pd.read_csv('sales_train.csv', parse_dates=['date'])
train['id'] = train['unique_id'].astype('str') + '_' + train['date'].astype('str')
train.set_index('id',inplace=True)
train = train[~train['sales'].isna()]   # drop rows with missing target
# Merge static product metadata; reset/set index restores original row order via .loc[train.index].
train = train.reset_index().merge(inventory, on='unique_id').set_index('id').loc[train.index]
# Merge per-day calendar features (holiday flags + before/after).
train = train.reset_index().merge(calendar, on=['date','warehouse']).set_index('id').loc[train.index]
fe_date(train)
fe_other(train)

# ---- TEST ----  (same pattern, but no target to filter on)
test = pd.read_csv('sales_test.csv', parse_dates=['date'])
test['id'] = test['unique_id'].astype('str') + '_' + test['date'].astype('str')
test.set_index('id',inplace=True)
test = test.reset_index().merge(inventory, on='unique_id').set_index('id').loc[test.index]
test = test.reset_index().merge(calendar, on=['date','warehouse']).set_index('id')
fe_date(test)
fe_other(test)

# ---- COMBINED FEATURES ----
# fe_combined needs both halves (e.g. for per-product mean price). It does NOT use `sales`, so this is leakage-safe.
all_data = pd.concat([train,test])
all_data = fe_combined(all_data)
train = all_data.loc[train.index]
test  = all_data.loc[test.index].drop(['sales','availability'],axis=1)


## 6. Sanity-check a single product's test rows

Always look at your data after a heavy merge/feature-engineering pipeline. Pick one `unique_id`,
sort by date, and eyeball: does it have the expected number of rows? Are the engineered columns
populated? Any unexpected NaNs?


In [ ]:
test.sort_values(by=['unique_id','date'], ascending=True)[test['unique_id']==1]


## 7. Split into features (X), target (y), and sample weights

Three things happen here:

1. **Split off the target.** `y_train = train['sales']`, then drop it from `X_train`.
2. **Stash `availability`.** It's a *historical-only* column (not present in test), so we can't use
   it as a feature for prediction, but we keep it aside in case we want it for diagnostics or
   sample weighting later.
3. **Per-product sample weights.** The competition metric is **weighted MAE** — different products
   carry different weights. We pull weights from `test_weights.csv` and broadcast them onto every
   row of training via a `unique_id → weight` map. LightGBM will use these so the loss focuses on
   the products that count most for the leaderboard.

> **Why weights matter:** If product A has weight 10 and product B has weight 1, a 1-unit error on
> A costs 10× as much as the same error on B. The model should "spend more attention" reducing A's
> errors. Without sample weights you'd be optimizing a metric the leaderboard isn't using.


In [ ]:
X_train = train.drop('sales',axis=1)
y_train = train['sales']
train_availability = X_train['availability']      # keep aside; we won't use it as a feature
X_train.drop('availability',inplace=True,axis=1)

# Per-product weights from the competition metric definition.
weights = pd.read_csv('test_weights.csv').set_index('unique_id')
X_train_weights = X_train['unique_id'].map(weights['weight'])  # broadcast weight to each row


## 8. Target-derived features (and the leakage trap)

We now build features that are **functions of historical sales**:

- `last_sales_ema005` — an exponentially-weighted moving average of past sales per product. The
  smoothing factor `α=0.005` corresponds to a very long memory (effective window ≈ `2/α - 1 ≈ 399` days).
  This is essentially a slowly-updating estimate of "typical demand for this product."
- `CN_sales_sum` — same smoothed signal, but summed across the family `(common_name, warehouse)`
  on each date. A family-level demand indicator.
- `last_sales_zs` — z-score of `last_sales_ema005` against the per-`(common_name, warehouse)`
  historical mean and std. "Is this product's recent demand high or low *for its peer group*?"

### Why are these in a separate cell?

Because they touch the **target**, they are uniquely vulnerable to **temporal leakage**:

- `.shift(1)` is essential — it makes sure today's EMA is computed using only sales **up to yesterday**.
- The frame is sorted chronologically before the EMA. `groupby(...).transform(lambda x: x.shift(1).ewm(...).mean())`
  computes the EMA per product, lagged by one row.
- If you're doing **time-based holdout validation** (training on `<2024-01` and validating on `≥2024-01`),
  these features must be computed using only data **before** the holdout cutoff for that fold. Otherwise
  your validation EMA "knows" the future.

> **Important caveat about the CV strategy in this notebook:** The actual training loop below uses
> `RepeatedKFold` (random splits), not a time-based split. With *target-derived* features built on the
> **full timeline**, every random fold's validation rows will have features that "saw" the future.

### Filtering early data

`X_train = X_train[X_train['date'] >= '2022-01-01']` drops pre-2022 rows.


In [ ]:
cat_cols = ['unique_id'] + list(X_train.columns[X_train.dtypes == 'object'])
all_data = pd.concat([X_train, test])
add_cols = ['last_sales_ema005','CN_sales_sum','last_sales_zs']

# Here there are a few additional features engineered from historical sales data.

# Build a continuous calendar per product spanning [first appearance .. last test date]
# so EMAs progress on every calendar day, not just sales days.
train_cp = train.groupby('unique_id')['date'].apply(lambda s: pd.date_range(s.min(), test.date.max())).explode().reset_index()
train_cp = train_cp.merge(
    pd.concat([train[['unique_id','date','sales','warehouse',]],
               test[['unique_id','date','warehouse']]]),
    on=['unique_id','date'],how='left')
train_cp = train_cp.merge(inventory, left_on='unique_id', right_index=True)
train_cp['common_name'] = train_cp['name'].apply(lambda x: x[:x.find('_')])
train_cp.sort_values('date',inplace=True)

# IMPORTANT: .shift(1) before .ewm() — today's EMA must use only sales up to yesterday.
train_cp['last_sales_ema005'] = train_cp.groupby(['unique_id'])['sales'].transform(lambda x: x.shift(1).ewm(alpha=.005).mean()).fillna(0)
# Family-level demand on each date/warehouse: sum of per-product EMAs.
train_cp['CN_sales_sum'] = train_cp.groupby(['common_name','warehouse','date'])['last_sales_ema005'].transform('sum')

# Merge the target-derived features back into all_data.
all_data = all_data.merge(train_cp.set_index(['unique_id','date'])[[
    'last_sales_ema005','CN_sales_sum'
]], left_on=['unique_id','date'],right_index=True,how='left')

# Z-score the EMA against the family's historical mean/std (computed on the train_cp continuous frame).
sales_stats = train_cp.groupby(['common_name','warehouse'])['sales'].agg(['mean','std'])
all_data['last_sales_zs'] = (all_data['last_sales_ema005'] - pd.MultiIndex.from_frame(all_data[['common_name','warehouse']]).map(
    sales_stats['mean']))/ pd.MultiIndex.from_frame(all_data[['common_name','warehouse']]).map(sales_stats['std'])

# Cutting all data prior to 2022 seems to help. This could be due to COVID effects, and also the fact that there is little data from the Germany warehouses before 2022.
X_train = X_train[X_train['date'] >= '2022-01-01']
y_train = y_train.loc[X_train.index]
X_train_weights = X_train_weights.loc[X_train.index]

# Attach the new target-derived features to X_train and test.
X_train[add_cols] = all_data[add_cols]
test[add_cols] = all_data[add_cols]

# Final cast: every categorical column → pandas 'category' dtype, which LightGBM consumes natively
# (no one-hot encoding, no label encoding by hand).
all_data[cat_cols] = all_data[cat_cols].astype('str').astype('category')


## 9. Hyperparameters

We define two parameter dicts: one for the model, one for the CV splitter.

### Model hyperparameters

- `learning_rate = 0.1` — moderate. Smaller lr + more trees usually generalizes better but trains
  more slowly; this notebook uses early stopping to avoid the cost of an excessively large `n_estimators`.
- `n_estimators = round(500/lr) = 5000` — a large *upper bound*. The actual number of trees used per
  fold is determined by **early stopping** (`es=10` rounds without validation improvement → stop).
- `objective='regression'`, `metric='rmse'` — standard L2 regression. We will report MAE on
  out-of-fold predictions, but training on RMSE often gives smoother trees.
- `reg_lambda=0`, `min_child_weight=1` — almost no regularization. Trees are constrained mostly by
  early stopping and the tree-growth defaults.
- `device_type='cpu'` — set to `'gpu'` only if your LightGBM build has GPU support compiled in;
  otherwise it errors out.

### CV hyperparameters

- `n_splits=3, n_repeats=1` — three folds, no repetition. Each row gets exactly one OOF prediction.
- Larger `n_repeats` would give a more stable OOF estimate (predictions averaged across multiple
  random partitions) at proportional compute cost.

### Why `RepeatedKFold` here, even though this is a time-series problem?

Pragmatic, not principled. Random k-fold mixes future and past in the same fold, so the OOF score
will be **optimistically biased** — features like rolling means and EMAs leak information that a
forward-only deployment wouldn't have. For a truly trustworthy validation score, use
`TimeSeriesSplit` and recompute target-derived features within each fold.


In [ ]:
lr = .1
es = 10                           # early-stopping patience: stop after `es` rounds without improvement
n_est = round(500/lr)             # generous upper bound; early stopping decides actual count
seed = 2

base_params = {
    'n_estimators':n_est
    ,'learning_rate':lr
    ,'verbose':-1                 # suppress LightGBM's chatter
    ,'random_state':seed
    ,'objective':'regression'     # L2 loss
    ,'metric':'rmse'
    ,'device_type':'cpu'          # requires LightGBM built with GPU support; set to 'cpu' if you don't have one
    ,'reg_lambda':0               # no L2 on leaf values
    ,'min_child_weight':1         # tiny leaves allowed
}
kf_params = {
    'n_splits':3
    ,'n_repeats':1
    ,'random_state':seed
}


## 10. The cross-validation training loop

This is the heart of the notebook. For each of the `n_splits × n_repeats` folds we:

1. Take the train rows in `idx_t` and the validation rows in `idx_v`.
2. Apply the **target power transform** (`y → y^0.5`) to both halves. Sales counts have a heavy
   right tail — a few products sell hundreds of units a day while most sell a handful. A square-root
   transform compresses the tail so the model isn't dominated by huge-volume items. We invert with
   `pred^(1/0.5) = pred^2` after prediction.
3. Fit a `LGBMRegressor` with **early stopping** on `(X_v, y_v)`. Note that this means the validation
   set leaks *very mildly* into model selection (we pick `best_iteration` from it). For competition
   scoring it's fine; for production, use a separate inner validation slice.
4. Predict `test` and `X_v`, undo the power transform, clip to `≥ 0` (sales can't be negative).
5. Store: per-fold test predictions (we'll average them) and OOF predictions (for honest validation).

### Predict the test set every fold

We predict `test` *inside* the fold loop because we want **`n_splits` different test predictions
to average together**. Averaging across folds is a cheap form of bagging that reduces variance,
typically improving leaderboard score.

### The dtype-cleanup block

LightGBM only accepts `int / float / bool / category` columns. The cleanup:

1. Detects any column that isn't already one of those (`_needs_cat`).
2. For each such column, computes the **union of values across train + test**, builds a
   `CategoricalDtype` from that union, and casts both frames to it. Sharing the same dtype object
   means the integer codes line up between train and test — critical so LightGBM treats
   "warehouse A" the same in both.
3. Re-applies the explicit `cat_cols` (e.g. `unique_id`, which is integer but should be treated
   categorically rather than ordinally — products 17 and 18 aren't "close").

Without this step you get either a `TypeError` from LightGBM or, worse, silently incorrect
predictions if codes don't align.

### Storing OOF predictions in a wide DataFrame

`oof_pred_df` has one column per repeat (`Pred_0`, ...). Within a single repeat, every row is in
exactly one validation fold, so each column is fully populated by row across folds. Averaging across
columns later gives the row's mean OOF prediction over repeats.


In [ ]:
drop_cols = ['date','name','L1_category_name_en']  # date is implicitly captured by 'days_since_2020' etc.; name was decomposed; L1 is too coarse
oof_preds = []
test_preds = []
pow_trans = True       # apply y -> y^0.5 power transform
pow_degree = .5

kf = RepeatedKFold(**kf_params)
X, y = deepcopy(X_train), deepcopy(y_train)   # work on copies; original frames stay clean
X[cat_cols] = all_data[cat_cols]              # bring in category dtypes
X.drop(drop_cols, axis=1, inplace=True)
test_copy = deepcopy(test)
test_copy[cat_cols] = all_data[cat_cols]
test_copy.drop(drop_cols, axis=1, inplace=True)

# LightGBM requires every feature column to be int / float / bool / category.
# Two things can go wrong here:
#   (1) cat_cols is built with `dtypes == 'object'`, which doesn't match
#       pandas' newer StringDtype, so str columns may slip through and never
#       get converted to category.
#   (2) Column assignment `X[cat_cols] = all_data[cat_cols]` can keep the
#       destination's existing dtype rather than adopting category.
# Fix both by detecting any non-numeric/bool/category column right before
# training and converting it to a CategoricalDtype whose levels are the union
# of train+test values, so X and test_copy share the same category codes.
import pandas.api.types as pdt
def _needs_cat(s):
    return not (pdt.is_numeric_dtype(s) or pdt.is_bool_dtype(s)
                or isinstance(s.dtype, pd.CategoricalDtype))
bad_cols = [c for c in X.columns if _needs_cat(X[c])]
for col in bad_cols:
    if col not in test_copy.columns:
        continue
    union = pd.unique(pd.concat([X[col], test_copy[col]], ignore_index=True).astype('object'))
    cat_dtype = pd.CategoricalDtype(categories=union)
    X[col] = X[col].astype('object').astype(cat_dtype)
    test_copy[col] = test_copy[col].astype('object').astype(cat_dtype)

# Also force the explicit cat_cols (e.g. unique_id, which is int but should
# be treated categorically) to all_data's dtype so they end up as category.
for col in [c for c in cat_cols if c in X.columns and isinstance(all_data[c].dtype, pd.CategoricalDtype)]:
    X[col] = X[col].astype(all_data[col].dtype)
    test_copy[col] = test_copy[col].astype(all_data[col].dtype)
print('dtypes after fix:', X.dtypes.value_counts().to_dict())

# OOF container: one column per CV repeat. Within a repeat, splits are disjoint, so
# every row gets filled exactly once.
oof_pred_df = pd.DataFrame(index=X.index, columns=['Pred_{0}'.format(i) for i in range(kf_params['n_repeats'])])

for i, (idx_t, idx_v) in enumerate(kf.split(X)):
    print(f"{i} - {len(idx_t)} - {len(idx_v)}")
    X_t, X_v = X.iloc[idx_t], X.iloc[idx_v]
    print(f"X_t cols: {X_t.columns.tolist()}")
    y_t, y_v = y.loc[X_t.index], y.loc[X_v.index]

    # Power transform compresses the heavy right tail of count-like targets.
    if pow_trans:
        y_t, y_v = np.power(y_t, pow_degree), np.power(y_v, pow_degree)

    lgbm = LGBMRegressor(**base_params)
    lgbm.fit(X_t, y_t, eval_set=[(X_v, y_v)],
             callbacks=[early_stopping(es), log_evaluation(100*es)])

    # Predict test, invert the power transform, clip negatives to zero (sales can't be negative).
    model_test_preds = np.power(lgbm.predict(test_copy).clip(0), 1/pow_degree) if pow_trans else lgbm.predict(test_copy).clip(0)
    test_preds.append(model_test_preds)

    # Predict OOF rows the same way and store in the wide frame.
    model_oof_preds = np.power(lgbm.predict(X_v).clip(0), 1/pow_degree) if pow_trans else lgbm.predict(X_v).clip(0)
    oof_pred_df.iloc[idx_v, int(i/kf_params['n_splits'])] = model_oof_preds

oof_preds.append(oof_pred_df)
print(len(test_preds))


## 11. Score the out-of-fold predictions

Now we evaluate the model on data it never saw during training (within each fold).

- `oof_pred_df` has one column per repeat → average across columns to get a single OOF prediction per row.
- We compute **weighted MAE** (the competition metric) using the per-product weights from earlier.
- This number is your **honest holdout score**, modulo the time-leakage caveat from the CV section.
  Treat it as an upper bound on test-set quality.

A useful diagnostic when iterating on features: log every `(feature_set, OOF_score)` pair and watch
the trend. If a fancy new feature improves training loss but worsens OOF, you've likely added noise
or leaked something subtly.


In [ ]:
oof_pred_df = pd.concat(oof_preds, axis=1)
test_pred_df = pd.DataFrame(np.transpose(test_preds), index=test.index)
oof_pred_vals = oof_pred_df.mean(axis=1)
np.round(mean_absolute_error(y_train, oof_pred_vals, sample_weight=X_train_weights), 3)


## 12. Build the submission file

Average test predictions across folds (bagging) and write to CSV. The column name `sales_hat` is
just convention; the competition expects whatever schema is documented on Kaggle.


In [ ]:
test_sub = test_pred_df.mean(axis=1)
test_sub.name = 'sales_hat'
test_sub.to_csv('lgbm_submission.csv')


## Recap — what to take away from this notebook

1. **Tabular forecasting ≠ classical time-series.** With panel data and rich exogenous features,
   GBDTs win. The skill becomes feature engineering, not picking an ARIMA order.
2. **Cyclical encoding (`sin`/`cos` of day-of-year)** lets the model treat seasonality as a continuous
   loop rather than a number line.
3. **Hierarchical features** (`common_name`, family-level aggregations) let products borrow strength
   from their siblings.
4. **Calendar proximity features** (day-before/day-after holiday) capture short-range demand spikes.
5. **Target-derived features must be lagged** (`.shift(1)` before `.ewm()`) and ideally
   recomputed inside each CV fold.
6. **`RepeatedKFold` on time-series data is a leakage hazard.** It's pragmatic, but it inflates
   the OOF score. For trustworthy validation use `TimeSeriesSplit` or a manual time-based holdout.
7. **Power-transforming a heavy-tailed target** (here `y → y^0.5`) often stabilizes regression.
   Always invert in the prediction step.
8. **Average test predictions across folds** for variance reduction (lightweight bagging).
9. **Sample weights** keep your loss aligned with the actual evaluation metric.
10. **LightGBM eats `category` dtype directly** — don't hand-encode unless you have to. But align
    train and test categorical levels (the dtype-cleanup block) or codes will silently disagree.

### Suggested exercises

- Replace `RepeatedKFold` with `TimeSeriesSplit` and recompute the OOF MAE. How much does it change?
- Add the integer `days_to_next_holiday` and `days_since_last_holiday` (instead of just the binary
  before/after flags). Does the OOF score improve?
- Try alternate target transforms: `log1p`, Box-Cox. Compare OOF MAE.
- Add lag features explicitly (sales 7 / 14 / 28 days ago) inside the time-aware CV loop.
- Tune `learning_rate` and `num_leaves` with Optuna or a coarse grid; see how much headroom is left.
